In [1]:
import matplotlib.pyplot as plt
import pickle

with open('training_history_v2.pkl', 'rb') as file:
    history = pickle.load(file)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Loss Curves')

plt.subplot(1,2,2)
plt.plot(history['val_acc'], label='Validation Accuracy')
plt.legend()
plt.title('Accuracy Curve')
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'training_history_v2.pkl'

In [16]:
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader, random_split
import torch
from torchvision.models import ResNet18_Weights
from PIL import Image
import pandas as pd

num_bins = ((2024 - 1875) // 15) + 1

class PreprocessedBuildingDataset(Dataset):
    def __init__(self, dataframe, preprocessed_dir, min_year=1875, max_year=2024, bin_width=15):
        """
        Args:
            dataframe (pd.DataFrame): DataFrame containing metadata (e.g., 'bouwjaar').
            preprocessed_dir (str): Path to the folder with preprocessed .pt files.
            min_year (int): The minimum year for binning.
            max_year (int): The maximum year for binning.
            bin_width (int): The width of each bin in years.
        """
        self.df = dataframe.reset_index(drop=True)
        self.preprocessed_dir = preprocessed_dir
        self.min_year = min_year
        self.max_year = max_year
        self.bin_width = bin_width
        self.num_bins = ((max_year - min_year) // bin_width) + 1

        # Add a 'label' column to the DataFrame
        self.df['label'] = ((self.df['bouwjaar'] - self.min_year) // self.bin_width).clip(0, self.num_bins - 1)
        
        # Extract labels as a NumPy array
        self.labels = self.df['label'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        """
        Returns:
            image (torch.Tensor): Preprocessed image tensor.
            label (torch.Tensor): Class label.
        """
        preprocessed_path = f"{self.preprocessed_dir}/{idx}.pt"
        try:
            data = torch.load(preprocessed_path, weights_only=True)
            image, label = data['image'], data['label']
        except Exception as e:
            print(f"Error loading file {preprocessed_path}: {e}")
            return torch.zeros((3, 224, 224)), -1  # Return placeholder in case of error

        return image, torch.tensor(label, dtype=torch.long)
    


# Load Data
df_loaded = pd.read_pickle('filtered_data.pkl')
preprocessed_dir = "preprocessed_v1"
full_dataset = PreprocessedBuildingDataset(df_loaded, preprocessed_dir)

# Split Dataset
train_size = int(0.8 * len(full_dataset))
val_size = (len(full_dataset) - train_size) // 2
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size]
)


def create_model(num_classes = num_bins):
    """
    Create a ResNet-18 model with frozen layers except the final fully connected layer.

    Args:
        num_classes (int): Number of output classes (e.g., number of bins).

    Returns:
        torch.nn.Module: A ResNet-18 model with customized final layer.
    """
    # Load ResNet-18 with pretrained weights (updated method)
    model = models.resnet18(weights=ResNet18_Weights.DEFAULT)

    # Freeze all layers
    for param in model.parameters():
        param.requires_grad = False

    # Replace the final fully connected layer
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    # Unfreeze the final fully connected layer
    for param in model.fc.parameters():
        param.requires_grad = True

    return model


model = create_model()


# Load best model
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Test evaluation
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

from tqdm import tqdm

correct = 0
total = 0

# Use tqdm for the test loop
with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="Evaluating", leave=True):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

# Print test accuracy
print(f'Test Accuracy: {correct / total:.4f}')


C:\Users\rkhaz\AppData\Local\Temp\ipykernel_45660\1903439578.py:102: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model.pth'))
Evalua

Test Accuracy: 0.4813


In [7]:
from torchvision import transforms
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader, random_split
import torch
from torchvision.models import ResNet18_Weights
from PIL import Image
import pandas as pd

import torch.nn.functional as F

def classify_image_with_time(model, image_path, device, transform, min_year, bin_width):
    """
    Classify a single image and map the predicted class to a time period, with a confidence score.

    Args:
        model: The trained PyTorch model.
        image_path: Path to the image to classify.
        device: Device (CPU or GPU) where the model is loaded.
        transform: Transformations applied to the input image.
        min_year: The start of the time period range.
        bin_width: The width of each time bin in years.

    Returns:
        predicted_class (int): The predicted class index.
        time_period (tuple): The corresponding time period (start_year, end_year).
        confidence_score (float): The confidence score of the prediction.
    """
    # Load and preprocess the image
    image = Image.open(image_path).convert("RGB")  # Convert to RGB if needed
    image = transform(image)  # Apply the same transformations used for training
    image = image.unsqueeze(0).to(device)  # Add batch dimension and move to device

    # Predict
    model.eval()
    with torch.no_grad():
        output = model(image)
        probabilities = F.softmax(output, dim=1)  # Convert logits to probabilities
        confidence_score, predicted_class = torch.max(probabilities, 1)
    
    # Map class to time period
    predicted_class = predicted_class.item()
    start_year = min_year + predicted_class * bin_width
    end_year = start_year + bin_width - 1
    time_period = (start_year, end_year)

    return predicted_class, time_period, confidence_score.item()



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Define the transform (same as training)
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def create_model(num_classes):
    """
    Create an EfficientNet-B0 model with dropout and a customized final layer.

    Args:
        num_classes (int): Number of output classes.

    Returns:
        torch.nn.Module: Customized EfficientNet-B0 model.
    """
    # Load EfficientNet-B0 with pretrained weights
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

    # Freeze all layers initially
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze the last two feature blocks (for more learning capacity)
    for param in model.features[-4:].parameters():
        param.requires_grad = True

    # Customize the final fully connected layer with dropout
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.5),
        nn.Linear(model.classifier[1].in_features, num_classes)
    )

    return model
# Initialize the model


import os

for picca in os.listdir('testing'):

    min_year=1900
    max_year=2024
    
    for bin_width in [10,7]:


        num_bins = ((max_year - min_year) // bin_width) + 1

      # Load the model with CPU mapping if no GPU is available

        if bin_width == 10:
            model = create_model(num_bins)
            model.load_state_dict(torch.load('best_model_10.pth', map_location=torch.device('cpu')))
            
        elif bin_width == 7:
            model = create_model(num_bins)
            model.load_state_dict(torch.load('best_model_7.pth', map_location=torch.device('cpu')))


        # Classify a single image
        image_path = f"testing/{picca}"  # Replace with your image path
        predicted_class, time_period, confidence_score = classify_image_with_time(
            model, image_path, device, transform, min_year, bin_width
        )

        print(f"{picca}: Predicted Time Period: {time_period[0]} - {time_period[1]} "
            f"with confidence: {confidence_score:.2%} for model with width: {bin_width}")



C:\Users\rkhaz\AppData\Local\Temp\ipykernel_14572\1415377302.py:112: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model_10.pth', map_

aker.png: Predicted Time Period: 2000 - 2009 with confidence: 64.74% for model with width: 10


C:\Users\rkhaz\AppData\Local\Temp\ipykernel_14572\1415377302.py:116: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model_7.pth', map_l

aker.png: Predicted Time Period: 1998 - 2004 with confidence: 35.66% for model with width: 7
allebe.png: Predicted Time Period: 2020 - 2029 with confidence: 32.83% for model with width: 10
allebe.png: Predicted Time Period: 2019 - 2025 with confidence: 19.32% for model with width: 7
boele.png: Predicted Time Period: 2010 - 2019 with confidence: 16.50% for model with width: 10
boele.png: Predicted Time Period: 1956 - 1962 with confidence: 15.73% for model with width: 7
bp.png: Predicted Time Period: 1930 - 1939 with confidence: 42.46% for model with width: 10
bp.png: Predicted Time Period: 1928 - 1934 with confidence: 36.39% for model with width: 7
centrum.png: Predicted Time Period: 1900 - 1909 with confidence: 74.07% for model with width: 10
centrum.png: Predicted Time Period: 1900 - 1906 with confidence: 44.74% for model with width: 7
hoofddorp.png: Predicted Time Period: 2000 - 2009 with confidence: 25.52% for model with width: 10
hoofddorp.png: Predicted Time Period: 2005 - 2011 wi

In [8]:
import os
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image

def classify_image_with_time(model, image_path, device, transform, min_year, bin_width):
    """
    Classify a single image and map the predicted class to a time period, with a confidence score.

    Args:
        model: The TorchScript model.
        image_path: Path to the image file.
        device: The device (CPU or GPU) on which to run inference.
        transform: Transformations applied to the input image.
        min_year: The starting year of the time period range.
        bin_width: The width (in years) of each time bin.

    Returns:
        predicted_class (int): The predicted class index.
        time_period (tuple): The corresponding time period (start_year, end_year).
        confidence_score (float): The confidence score for the prediction.
    """
    # Load and preprocess the image.
    image = Image.open(image_path).convert("RGB")
    image = transform(image)         # Apply training transformations.
    image = image.unsqueeze(0).to(device)  # Add batch dimension and move to device.

    model.eval()
    with torch.no_grad():
        output = model(image)
        # Convert logits to probabilities.
        probabilities = F.softmax(output, dim=1)
        confidence_score, predicted_class = torch.max(probabilities, 1)

    predicted_class = predicted_class.item()
    start_year = min_year + predicted_class * bin_width
    end_year = start_year + bin_width - 1
    return predicted_class, (start_year, end_year), confidence_score.item()

# Set up the device.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define the transform (exactly as used in training).
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load the TorchScript model.
# Make sure that 'model_1.pt' is in the current working directory (or provide the correct path).
model = torch.jit.load("model_1.pt", map_location=device)

# Define the time period parameters.
min_year = 1900
bin_width = 7  # Adjust this if your TorchScript model was generated with a different bin width.

# Loop over images in the 'testing' folder and classify each one.
for pic in os.listdir('testing'):
    image_path = os.path.join('testing', pic)
    predicted_class, time_period, confidence_score = classify_image_with_time(
        model, image_path, device, transform, min_year, bin_width
    )
    print(f"{pic}: Predicted Time Period: {time_period[0]} - {time_period[1]} "
          f"with confidence: {confidence_score:.2%} for bin width: {bin_width}")


aker.png: Predicted Time Period: 1998 - 2004 with confidence: 35.66% for bin width: 7
allebe.png: Predicted Time Period: 2019 - 2025 with confidence: 19.32% for bin width: 7
boele.png: Predicted Time Period: 1956 - 1962 with confidence: 15.73% for bin width: 7
bp.png: Predicted Time Period: 1928 - 1934 with confidence: 36.39% for bin width: 7
centrum.png: Predicted Time Period: 1900 - 1906 with confidence: 44.74% for bin width: 7
hoofddorp.png: Predicted Time Period: 2005 - 2011 with confidence: 23.85% for bin width: 7
huis.png: Predicted Time Period: 1991 - 1997 with confidence: 41.30% for bin width: 7
oma.png: Predicted Time Period: 1900 - 1906 with confidence: 39.49% for bin width: 7
overkant.jpg: Predicted Time Period: 1956 - 1962 with confidence: 16.51% for bin width: 7
sloterweg.png: Predicted Time Period: 2019 - 2025 with confidence: 50.46% for bin width: 7
stalk.png: Predicted Time Period: 1942 - 1948 with confidence: 19.81% for bin width: 7


13